# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")
print(f"Temporal Coverage: {metadata.temporal_coverage}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

In [ ]:
# List all available RecordSets, their @id, and contained fields
record_sets = list(dataset.metadata.record_sets)
print(f"\nFound {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, dataType: {getattr(f, 'data_type', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from selected record set(s) into pandas DataFrames for analysis.
Each `record_set` and field is referenced by its `@id`.

Below, select the primary record set (`@id`) for analysis. If there are multiple record sets of interest, you can add more to the list.

In [ ]:
# Collect RecordSet @ids (customize as needed based on previous cell's output)
record_sets_for_analysis = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_sets_for_analysis:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
        print()
    else:
        print(f"[Info] No records found for RecordSet: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and categorizing data.
All fields should be referenced via their `@id`.

> **Tip:** If unsure which numeric fields exist, preview available columns above.

In [ ]:
# For demonstration, pick the first available DataFrame and a likely numeric column (edit as needed)
if dataframes:
    # Select a record set
    first_rs_id = next(iter(dataframes.keys()))
    df = dataframes[first_rs_id]
    
    # Identify likely numeric columns
    numeric_cols = df.select_dtypes(include=['number', 'float64', 'int64']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # or pick manually by @id from above
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing first 5):")
        print(filtered_df.head())

        # Normalization
        if filtered_df.shape[0] > 0:
            filtered_df = filtered_df.copy()
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            )
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Attempt to group by a non-numeric field
            group_field = None
            for col in df.columns:
                if col != numeric_field_id and df[col].dtype == 'object':
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
                print(grouped_df.head())
            else:
                print("No suitable non-numeric field for grouping found.")
        else:
            print("No records above threshold for normalization and grouping.")
    else:
        print("No numeric columns detected for EDA.")
else:
    print("No dataframes available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram or barplot depending on data type
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Use numeric_field_id from the previous EDA cell
    if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # If grouping field found, show barplot
        if 'group_field' in locals() and group_field:
            group_agg = df.groupby(group_field)[numeric_field_id].mean().sort_values()
            plt.figure(figsize=(8, 4))
            sns.barplot(x=group_agg.index, y=group_agg.values)
            plt.title(f"Mean {numeric_field_id} by {group_field}")
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xlabel(f"{group_field}")
            plt.xticks(rotation=45, ha='right')
            plt.show()
    else:
        print("No suitable numeric field for visualization.")

## 6. Conclusion
This notebook provided:
- Loading and overview of the FAIR^2 dataset defined by a Croissant schema via the `mlcroissant` library.
- Exploration of available record sets and fields by `@id`.
- Loading and simple exploratory processing of tabular data.
- Normalization and basic grouping for selected numeric fields.
- Example visualizations of data distributions and relationships.

**Continue your analysis** by inspecting additional fields, combining record sets, or building statistical and predictive models. For full dataset provenance and context, see [the accompanying FAIR^2 record](https://doi.org/10.71728/senscience.y7m0-f273).